# U decomposition analysis

Generate Haar-random `SU(n)` matrices, decompose them with `unitary_to_G_rotations`, and convert the returned two-level rotations into ion-pulse parameters.

Notes:
- `mode="target"` converts the returned elimination rotations into the pulse sequence that synthesizes the target `U`.
- Each step returns `coupling`, `theta`, `phi`, and an extra diagonal phase `gamma`. `gamma` is the additional addressed-pair phase that must be tracked explicitly.
- For the `haar_su` sampler below, the residual `V` is numerically the identity. For a more general `U(n)` input, a residual diagonal can remain.
- Set `n = 29` when you want the active Ba-137 manifold size instead of a small test case.


In [12]:
import numpy as np
from numpy.linalg import norm

from U_decomp import unitary_to_G_rotations

np.set_printoptions(precision=4, suppress=True)


In [13]:
def haar_unitary(n, rng=None):
    rng = np.random.default_rng(rng)
    X = (rng.standard_normal((n, n)) + 1j * rng.standard_normal((n, n))) / np.sqrt(2)
    Q, R = np.linalg.qr(X)
    d = np.diag(R)
    D = d / np.abs(d)
    return Q * D


def haar_su(n, rng=None):
    U = haar_unitary(n, rng)
    phi = np.angle(np.linalg.det(U)) / n
    return U * np.exp(-1j * phi)


In [14]:
def wrap_pi(x):
    return (x + np.pi) % (2 * np.pi) - np.pi


def ion_pulse_unitary(coupling, theta, phi, dim):
    i, j = coupling
    U = np.eye(dim, dtype=complex)
    c = np.cos(theta / 2)
    s = np.sin(theta / 2)
    U[i, i] = c
    U[j, j] = c
    U[i, j] = -1j * np.exp(1j * phi) * s
    U[j, i] = -1j * np.exp(-1j * phi) * s
    return U


def phase_update_unitary(coupling, gamma, dim):
    i, j = coupling
    D = np.eye(dim, dtype=complex)
    D[i, i] = np.exp(1j * gamma)
    D[j, j] = np.exp(-1j * gamma)
    return D


def rotation_matrix_to_pulse_parameters(G, tol=1e-10):
    G = np.asarray(G, dtype=complex)
    if G.ndim != 2 or G.shape[0] != G.shape[1]:
        raise ValueError("G must be square")

    dim = G.shape[0]
    candidates = [(p, q) for p in range(dim) for q in range(p + 1, dim)
                  if abs(G[p, q]) > tol or abs(G[q, p]) > tol]
    if len(candidates) != 1:
        raise ValueError(f"Expected exactly one coupled pair, found {candidates}")

    i, j = candidates[0]
    mask = np.ones_like(G, dtype=bool)
    mask[[i, j], :] = False
    mask[:, [i, j]] = False
    if norm(G[mask] - np.eye(dim, dtype=complex)[mask]) > 100 * tol:
        raise ValueError("Extra couplings detected outside the active 2x2 block")

    c = G[i, i]
    s = G[i, j]
    gamma = float(wrap_pi(np.angle(c)))
    theta = float(2.0 * np.arctan2(np.clip(np.abs(s), 0.0, 1.0), np.clip(np.abs(c), 0.0, 1.0)))
    phi = float((np.angle(s) - gamma + np.pi / 2) % (2 * np.pi))

    coupling = (i, j)
    reconstructed = phase_update_unitary(coupling, gamma, dim) @ ion_pulse_unitary(coupling, theta, phi, dim)

    return {
        "coupling": coupling,
        "theta": theta,
        "theta_over_pi": float(theta / np.pi),
        "phi": phi,
        "gamma": gamma,
        "reconstruction_error": float(norm(reconstructed - G)),
    }


def rotations_to_ion_schedule(rotation_mats, mode="target", tol=1e-10):
    if mode not in {"target", "elimination"}:
        raise ValueError("mode must be 'target' or 'elimination'")
    if len(rotation_mats) == 0:
        return []

    if mode == "target":
        sequence = [G.conj().T for G in rotation_mats[::-1]]
    else:
        sequence = list(rotation_mats)

    dim = sequence[0].shape[0]
    z_frame = np.zeros(dim, dtype=float)
    schedule = []

    for step_idx, G in enumerate(sequence, start=1):
        step = rotation_matrix_to_pulse_parameters(G, tol=tol)
        i, j = step["coupling"]
        gamma = step["gamma"]
        z_frame[i] += gamma
        z_frame[j] -= gamma

        step["step"] = step_idx
        step["z_phase_update"] = {i: gamma, j: -gamma}
        step["z_frame_after"] = z_frame.copy()
        schedule.append(step)

    return schedule


def schedule_to_lists(schedule):
    couplings = [step["coupling"] for step in schedule]
    thetas = [step["theta"] for step in schedule]
    phis = [step["phi"] for step in schedule]
    gammas = [step["gamma"] for step in schedule]
    return couplings, thetas, phis, gammas


def unitary_from_ion_schedule(couplings, thetas, phis, dim, gammas=None, return_z_frame=False):
    if not (len(couplings) == len(thetas) == len(phis)):
        raise ValueError("couplings, thetas, and phis must have the same length")
    if gammas is not None and len(gammas) != len(couplings):
        raise ValueError("gammas must match the number of pulses")

    U = np.eye(dim, dtype=complex)
    z_frame = np.zeros(dim, dtype=float)

    for idx, (coupling, theta, phi) in enumerate(zip(couplings, thetas, phis)):
        step_unitary = ion_pulse_unitary(coupling, theta, phi, dim)

        if gammas is not None:
            gamma = gammas[idx]
            step_unitary = phase_update_unitary(coupling, gamma, dim) @ step_unitary
            i, j = coupling
            z_frame[i] += gamma
            z_frame[j] -= gamma

        U = step_unitary @ U

    if return_z_frame:
        return U, z_frame
    return U


In [23]:
n = 8  # set to 29 for the active Ba-137 manifold
rng = 1234
center = 0

U = haar_su(n, rng=rng)

Hadamard = np.array([[1, 1], [1, -1]]) / np.sqrt(2)
Hadamard_2 = np.kron(Hadamard, Hadamard)
Hadamard_3 = np.kron(Hadamard_2, Hadamard)
U = Hadamard_3 
rotation_mats, V = unitary_to_G_rotations(U, center=center)
schedule = rotations_to_ion_schedule(rotation_mats, mode="target")
couplings, thetas, phis, gammas = schedule_to_lists(schedule)

print(f"n = {n}, center = {center}")
print(f"number of TAQR rotations: {len(rotation_mats)}")
print(f"||V - I|| = {norm(V - np.eye(n)):.3e}")
print("\nTarget-side pulse schedule:")

for step in schedule:
    i, j = step["coupling"]
    gamma = step["gamma"]
    if abs(gamma) < 1e-12:
        phase_note = "no extra diagonal phase"
    else:
        phase_note = f"extra phase: level {i} += {gamma:+.6f}, level {j} += {-gamma:+.6f}"

    print(
        f"step {step['step']:2d}: coupling={step['coupling']}, "
        f"theta={step['theta']:.6f} rad ({step['theta_over_pi']:.6f} pi), "
        f"phi={step['phi']:.6f} rad, gamma={gamma:+.6f} rad, {phase_note}"
    )

print("\nCopy/paste lists:")
print("couplings =", couplings)
print("thetas    =", thetas)
print("phis      =", phis)
print("gammas    =", gammas)


n = 8, center = 0
number of TAQR rotations: 28
||V - I|| = 6.185e-16

Target-side pulse schedule:
step  1: coupling=(0, 1), theta=1.663093 rad (0.529379 pi), phi=1.570796 rad, gamma=-3.141593 rad, extra phase: level 0 += -3.141593, level 1 += +3.141593
step  2: coupling=(0, 2), theta=2.766372 rad (0.880564 pi), phi=4.712389 rad, gamma=-3.141593 rad, extra phase: level 0 += -3.141593, level 2 += +3.141593
step  3: coupling=(0, 1), theta=2.354848 rad (0.749571 pi), phi=4.712389 rad, gamma=+0.000000 rad, no extra diagonal phase
step  4: coupling=(0, 3), theta=2.785864 rad (0.886768 pi), phi=1.570796 rad, gamma=+0.000000 rad, no extra diagonal phase
step  5: coupling=(0, 2), theta=0.863912 rad (0.274992 pi), phi=1.570796 rad, gamma=+0.000000 rad, no extra diagonal phase
step  6: coupling=(0, 1), theta=1.084569 rad (0.345229 pi), phi=1.570796 rad, gamma=-3.141593 rad, extra phase: level 0 += -3.141593, level 1 += +3.141593
step  7: coupling=(0, 4), theta=2.457340 rad (0.782196 pi), phi=4.71

In [24]:
U_pulses_only = unitary_from_ion_schedule(couplings, thetas, phis, dim=n)
U_with_gammas, z_frame = unitary_from_ion_schedule(
    couplings, thetas, phis, dim=n, gammas=gammas, return_z_frame=True
)

print("max per-step reconstruction error:", max(step['reconstruction_error'] for step in schedule))
print("pulse-only error vs target U:", norm(U_pulses_only - U))
print("pulse+gamma error vs target U:", norm(U_with_gammas - U))
print("final accumulated z-frame:")
print(z_frame)


max per-step reconstruction error: 4.0096103516872613e-16
pulse-only error vs target U: 3.8732755140678865
pulse+gamma error vs target U: 9.71617245449103e-16
final accumulated z-frame:
[-18.8496   9.4248   3.1416   0.       3.1416   0.       0.       3.1416]


## Gamma minimization and virtual-Z compilation

Allowing `theta` to run from `0` to `2*pi` gives a second exact branch for every step:

- original branch: `(theta, phi, gamma)`
- alternate branch: `(2*pi - theta, phi + pi, wrap_pi(gamma + pi))`

That alternate branch cannot eliminate `gamma` in general, but it can reduce `|gamma|` to at most `pi/2` for each step.

After that, the remaining `gamma` values can be folded into a running virtual-Z frame. With the convention used here, the exact compiled unitary is

`U_target = D_final @ U_pulses`

where `U_pulses` is built from the programmed LO phases and `D_final = diag(exp(1j * z_final))` is the final virtual-Z frame.


In [25]:
def minimize_gamma_for_step(step):
    alternate = dict(step)
    alternate["theta"] = float(2 * np.pi - step["theta"])
    alternate["theta_over_pi"] = float(alternate["theta"] / np.pi)
    alternate["phi"] = float((step["phi"] + np.pi) % (2 * np.pi))
    alternate["gamma"] = float(wrap_pi(step["gamma"] + np.pi))
    return alternate if abs(alternate["gamma"]) < abs(step["gamma"]) else dict(step)


def minimize_schedule_gamma(schedule):
    return [minimize_gamma_for_step(step) for step in schedule]


def virtual_z_diagonal(z_frame):
    return np.diag(np.exp(1j * np.asarray(z_frame, dtype=float)))


def compile_virtual_z_schedule(schedule):
    if len(schedule) == 0:
        return [], np.array([])

    if "z_frame_after" in schedule[0]:
        dim = len(schedule[0]["z_frame_after"])
    else:
        dim = max(max(step["coupling"]) for step in schedule) + 1

    frame = np.zeros(dim, dtype=float)
    compiled = []

    for step in schedule:
        i, j = step["coupling"]
        programmed_phi = float((step["phi"] - (frame[i] - frame[j])) % (2 * np.pi))

        compiled_step = dict(step)
        compiled_step["frame_before"] = frame.copy()
        compiled_step["phi_programmed"] = programmed_phi

        frame[i] += step["gamma"]
        frame[j] -= step["gamma"]

        compiled_step["frame_after"] = frame.copy()
        compiled.append(compiled_step)

    return compiled, frame


def programmed_schedule_to_unitary(couplings, thetas, programmed_phis, dim, final_frame=None):
    U_pulses = unitary_from_ion_schedule(couplings, thetas, programmed_phis, dim)
    if final_frame is None:
        return U_pulses
    return virtual_z_diagonal(final_frame) @ U_pulses


In [ ]:
schedule_min = minimize_schedule_gamma(schedule)
compiled_schedule, z_final = compile_virtual_z_schedule(schedule_min)

couplings_vz = [step["coupling"] for step in compiled_schedule]
thetas_vz = [step["theta"] for step in compiled_schedule]
phis_vz = [step["phi_programmed"] for step in compiled_schedule]
gammas_vz = [step["gamma"] for step in compiled_schedule]

print("max |gamma| before minimization:", max(abs(step["gamma"]) for step in schedule))
print("max |gamma| after minimization :", max(abs(step["gamma"]) for step in schedule_min))
print("\nVirtual-Z compiled schedule:")

couplings = []
thetas = []
phases = []

for step in compiled_schedule:
    print(
        f"step {step['step']:2d}: coupling={step['coupling']}, "
        f"theta={step['theta']:.6f} rad ({step['theta_over_pi']:.6f} pi), "
        f"phi_programmed={step['phi_programmed']:.6f} rad, gamma={step['gamma']:+.6f} rad"
    )
    couplings.append(step["coupling"])
    thetas.append(step["theta"])
    phases.append(step["phi_programmed"])

print("\nProgrammed LO phases:", phis_vz)
print("Final virtual-Z frame:")
print(z_final)

U_programmed_only = programmed_schedule_to_unitary(couplings_vz, thetas_vz, phis_vz, dim=n)
U_programmed_exact = programmed_schedule_to_unitary(couplings_vz, thetas_vz, phis_vz, dim=n, final_frame=z_final)

print("\nprogrammed-pulses-only error vs target U:", norm(U_programmed_only - U))
print("programmed-pulses + final virtual-Z error vs target U:", norm(U_programmed_exact - U))


max |gamma| before minimization: 3.141592653589793
max |gamma| after minimization : 0.0

Virtual-Z compiled schedule:
step  1: coupling=(0, 1), theta=4.620092 rad (1.470621 pi), phi_programmed=4.712389 rad, gamma=+0.000000 rad
step  2: coupling=(0, 2), theta=3.516813 rad (1.119436 pi), phi_programmed=1.570796 rad, gamma=+0.000000 rad
step  3: coupling=(0, 1), theta=2.354848 rad (0.749571 pi), phi_programmed=4.712389 rad, gamma=+0.000000 rad
step  4: coupling=(0, 3), theta=2.785864 rad (0.886768 pi), phi_programmed=1.570796 rad, gamma=+0.000000 rad
step  5: coupling=(0, 2), theta=0.863912 rad (0.274992 pi), phi_programmed=1.570796 rad, gamma=+0.000000 rad
step  6: coupling=(0, 1), theta=5.198616 rad (1.654771 pi), phi_programmed=4.712389 rad, gamma=+0.000000 rad
step  7: coupling=(0, 4), theta=3.825845 rad (1.217804 pi), phi_programmed=1.570796 rad, gamma=+0.000000 rad
step  8: coupling=(0, 3), theta=0.942546 rad (0.300022 pi), phi_programmed=4.712389 rad, gamma=+0.000000 rad
step  9: c

In [29]:
print(sum(thetas)/np.pi)

13.013952711413868
